# Notebook 03: Estimation, bivariate analysis and regression

**Introduction to AI-Assisted Research Methods and Data Analysis** · Day 3

NexusHealth · Adamu Onu

You will choose methods that match the question, compare groups with effect sizes and confidence intervals, and fit linear and logistic regression models to estimate the association between medication adherence and blood pressure, adjusting for confounders.

> **Teaching data only.** Every patient in this course is synthetic. The associations are simulated and are not clinical evidence. Never paste identifiable patient data, confidential manuscripts or restricted full text into a public AI tool.

### How to use this notebook

1. Run cells from top to bottom with **Shift + Enter**.
2. Before you run a cell, say out loud (or write) what you expect it to do. After it runs, check whether you were right. We call this *predict, run, inspect*.
3. Cells marked **Exercise** have gaps for you to fill. Cells marked **AI assist** suggest a prompt for GitHub Copilot Chat (or your own assistant).
4. If things get tangled, use **Restart** then **Run All** from the notebook toolbar.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the course folder, wherever this notebook lives inside it.
ROOT = Path.cwd()
while not (ROOT / "data" / "raw").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Course folder not found. Open the whole course folder in VS Code."

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_REFERENCE = ROOT / "data" / "reference"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"
for folder in (DATA_PROCESSED, OUT_TABLES, OUT_FIGURES):
    folder.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
print("Course folder found:", ROOT.name)

In [ ]:
analysis_path = DATA_PROCESSED / "analysis_set.csv"
if not analysis_path.exists():
    analysis_path = DATA_REFERENCE / "analysis_set_reference.csv"
    print("Your Day 2 file was not found, so the reference analysis set is used instead.")
df = pd.read_csv(analysis_path, parse_dates=["visit_date"])

# Tell Python which variables are categories, and which level is the reference.
df["sex"] = pd.Categorical(df["sex"], categories=["Female", "Male"])
df["smoking_status"] = pd.Categorical(df["smoking_status"], categories=["Never", "Former", "Current"])
df["clinic"] = pd.Categorical(df["clinic"], categories=["North", "East", "South", "West"])
print(f"Loaded {analysis_path.name}: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

sns.set_theme(style="whitegrid")
BRAND = "#002366"

# Part A: Estimation and method selection (block 1)

Before choosing a test, write down the **estimand**: the quantity you want to know.

> *Among adults in primary care, how much does mean systolic BP differ per 10-point higher adherence score, among patients of the same age, sex, BMI, diabetes status and smoking status?*

Then the method follows from the type of outcome and exposure:

| Outcome | Exposure | Bivariate method | Effect measure | Multivariable model |
|---|---|---|---|---|
| Continuous (SBP) | Two groups | Welch t-test | Difference in means | Linear regression |
| Continuous | Continuous | Correlation | Pearson r (or Spearman) | Linear regression |
| Binary (uncontrolled BP) | Categories | Chi-square test | Difference in proportions | Logistic regression |
| Binary | Continuous | (go straight to a model) | Odds ratio | Logistic regression |

**A confidence interval** gives the range of values for the true effect that are reasonably compatible with the data. **A p-value** tells you how surprising the data would be if there were truly no effect. Neither tells you whether the effect matters clinically; that is your call.

## A1. A mean with a 95% confidence interval

The interval is the estimate plus or minus about two standard errors.

In [ ]:
sbp = df["systolic_bp"]
mean = sbp.mean()
se = sbp.std() / np.sqrt(len(sbp))
t_crit = stats.t.ppf(0.975, df=len(sbp) - 1)
print(f"Mean SBP {mean:.1f} mmHg, 95% CI {mean - t_crit * se:.1f} to {mean + t_crit * se:.1f} (n = {len(sbp)})")

## Exercise 3.0: Choose the method

For each question, name the outcome type, the exposure type, a suitable bivariate method and the effect measure you would report.

1. Is mean BMI different between current smokers and never smokers?
2. Is the proportion with uncontrolled BP different between clinics?
3. Is higher adherence associated with uncontrolled BP after allowing for age and diabetes?

**Your answer:** _Double-click here and write your answer._

# Part B: Bivariate analysis (block 2)

## B1. Comparing two means: SBP by sex

Always start with a description and a picture, then the test.

In [ ]:
display(df.groupby("sex", observed=True)["systolic_bp"].agg(["count", "mean", "std", "median"]).round(1))

fig, ax = plt.subplots(figsize=(6, 3.6))
sns.boxplot(data=df, x="sex", y="systolic_bp", color="#DEE6EF", ax=ax)
ax.set(xlabel="", ylabel="SBP (mmHg)", title="Systolic BP by sex")
plt.show()

We use **Welch's t-test**, which does not assume the two groups have equal variances. It is a safe default. We report the **difference in means with its 95% CI**, not just the p-value.

In [ ]:
male = df.loc[df["sex"] == "Male", "systolic_bp"]
female = df.loc[df["sex"] == "Female", "systolic_bp"]
welch = stats.ttest_ind(male, female, equal_var=False)
ci = welch.confidence_interval(0.95)
print(f"Difference (male minus female): {male.mean() - female.mean():.1f} mmHg, "
      f"95% CI {ci.low:.1f} to {ci.high:.1f}, p = {welch.pvalue:.3f}")

## Exercise 3A (part 1): SBP by diabetes

Compare systolic BP between patients with and without diabetes. Give group sizes and means, a boxplot, and the difference (diabetes minus no diabetes) with 95% CI and p-value.

**AI assist.** Ask Copilot to *explain* the Welch test output line by line. Then check which group was subtracted from which. Getting the direction wrong is one of the most common errors.

In [ ]:
# Group summaries
...

# Boxplot
...

# Welch t-test: diabetes minus no diabetes
...

**Interpret in one or two sentences, and give one limitation.**

**Your answer:** _Double-click here and write your answer._

## B2. Comparing proportions: uncontrolled BP by smoking status

For two categorical variables, start with a cross-table showing **row percentages**, then use a chi-square test. Check the *expected* counts: the test is unreliable if many are below 5.

In [ ]:
table = pd.crosstab(df["smoking_status"], df["uncontrolled_bp"])
display(table)
display((pd.crosstab(df["smoking_status"], df["uncontrolled_bp"], normalize="index") * 100).round(1))

chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"Chi-square = {chi2:.2f}, df = {dof}, p = {p:.3f}; smallest expected count = {expected.min():.1f}")

## Exercise 3A (part 2): Uncontrolled BP by diabetes

Make the cross-table with row percentages, run the chi-square test, and report the **difference in proportions** (diabetes minus no diabetes) in percentage points.

Note: for a 2 x 2 table both Python and R apply a small *continuity correction* by default.

In [ ]:
# Cross-table and row percentages
...

# Chi-square test
...

# Difference in proportions (percentage points)
...

## B3. Correlation: adherence and SBP

Plot first. A correlation coefficient summarises a *straight-line* relationship in one number between -1 and 1. Spearman's version uses ranks and is less affected by outliers.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.regplot(data=df, x="medication_adherence", y="systolic_bp", ax=ax,
            scatter_kws={"alpha": 0.4, "color": BRAND}, line_kws={"color": "#A2B5CD"})
ax.set(xlabel="Adherence score (%)", ylabel="SBP (mmHg)", title="Adherence and systolic BP")
plt.show()

pearson = stats.pearsonr(df["medication_adherence"], df["systolic_bp"])
spearman = stats.spearmanr(df["medication_adherence"], df["systolic_bp"])
print(f"Pearson r = {pearson.statistic:.2f} (p = {pearson.pvalue:.3f}); "
      f"Spearman rho = {spearman.statistic:.2f} (p = {spearman.pvalue:.3f})")

A correlation of about -0.13 is weak: adherence explains under 2% of the variation in SBP. It can still matter, because a small shift in the average BP of a whole population can be clinically important. Correlation also cannot adjust for confounders, which is why we now turn to regression.

# Part C: Linear regression (block 3)

Linear regression estimates how the **mean** of a continuous outcome changes with each predictor, holding the other predictors constant.

`SBP = intercept + b1 x adherence_10 + b2 x age + ... + error`

* `b1` is the difference in mean SBP for a 10-point higher adherence score, among people with the same values of the other variables.
* For categories (sex, smoking), each coefficient compares a level with the **reference level** (Female, Never).

Which covariates? Those on our Day 1 framework map as plausible **confounders** (they affect both adherence and BP). We do not adjust for mediators or colliders.

## C1. The unadjusted (crude) model

In [ ]:
crude_lm = smf.ols("systolic_bp ~ adherence_10", data=df).fit()
print(crude_lm.summary().tables[1])

**Reading the output.** The intercept is the predicted SBP at an adherence of 0 (rarely useful by itself). The `adherence_10` coefficient is the one that answers our question.

## Exercise 3B: Fit, check and interpret an adjusted linear model

1. Fit the adjusted model with `adherence_10`, `age`, `sex`, `bmi`, `diabetes` and `smoking_status`. Wrap diabetes in `C()` so Python treats it as a category.
2. Make a small table comparing the adherence estimate and 95% CI in the crude and adjusted models.

In [ ]:
adjusted_lm = smf.ols("systolic_bp ~ ...", data=df).fit()
print(adjusted_lm.summary().tables[1])

# Compare the adherence estimate in the two models
...

**Interpret the adjusted estimate** in one sentence a colleague could read, then say why it changed from the crude estimate.

**Your answer:** _Double-click here and write your answer._

## C2. Check the model's assumptions

Linear regression assumes the residuals (observed minus predicted) are roughly symmetric, have similar spread across predicted values, and have no strong pattern. Two plots cover most of this.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(adjusted_lm.fittedvalues, adjusted_lm.resid, alpha=0.4, color=BRAND)
axes[0].axhline(0, color="grey", linestyle="--")
axes[0].set(xlabel="Fitted SBP (mmHg)", ylabel="Residual (mmHg)", title="Residuals vs fitted")
sm.qqplot(adjusted_lm.resid, line="s", ax=axes[1], markerfacecolor=BRAND, markeredgecolor=BRAND, alpha=0.4)
axes[1].set(title="Normal Q-Q plot of residuals")
fig.tight_layout()
fig.savefig(OUT_FIGURES / "fig_linear_diagnostics.png", dpi=300)
plt.show()
print(f"R-squared: {adjusted_lm.rsquared:.3f}")

**Diagnostic note.** In two sentences, say what the plots show and whether it changes your conclusion.

**Your answer:** _Double-click here and write your answer._

# Part D: Logistic regression (block 4)

For a binary outcome (uncontrolled BP yes/no), logistic regression models the **log-odds** of the outcome. Exponentiating a coefficient gives an **odds ratio (OR)**.

| Scale | Meaning | Example |
|---|---|---|
| Probability (risk) | Share of people with the outcome | 0.69 = 69% |
| Odds | Probability / (1 - probability) | 0.69 / 0.31 = 2.2 |
| Odds ratio | Odds in one group / odds in another | OR 0.8 = odds 20% lower |

> **Odds are not risks.** When the outcome is common (here about 69%), an OR of 0.8 does **not** mean a 20% lower risk; the change in risk is smaller. Use predicted probabilities to talk about risk.

In [ ]:
crude_logit = smf.logit("uncontrolled_bp ~ adherence_10", data=df).fit(disp=False)
print(crude_logit.summary().tables[1])
print("Crude OR per 10 points:", round(np.exp(crude_logit.params["adherence_10"]), 3))

## Exercise 3C: Write a defensible logistic interpretation

1. Fit the adjusted logistic model with the same covariates as the linear model.
2. Show odds ratios with 95% CIs.
3. Calculate the average predicted probability of uncontrolled BP if everyone had an adherence score of 40, 60 and 80 (keeping their other characteristics as they are).

In [ ]:
# 1. Adjusted model
adjusted_logit = ...

# 2. Odds ratios with 95% CI
...

# 3. Average predicted probabilities at adherence 40, 60 and 80
...

**Write three sentences:** the adjusted OR with its CI, what it means in words, and what the predicted probabilities add.

**Your answer:** _Double-click here and write your answer._

## D2. Save the model results for Day 4

In [ ]:
def tidy_model(model, label, exponentiate=False):
    ci = model.conf_int()
    out = pd.DataFrame({"model": label, "term": model.params.index, "estimate": model.params.values,
                        "ci_low": ci[0].values, "ci_high": ci[1].values, "p_value": model.pvalues.values,
                        "n": int(model.nobs)})
    if exponentiate:
        out[["estimate", "ci_low", "ci_high"]] = np.exp(out[["estimate", "ci_low", "ci_high"]])
    return out

results = pd.concat([
    tidy_model(crude_lm, "Linear, crude"),
    tidy_model(adjusted_lm, "Linear, adjusted"),
    tidy_model(crude_logit, "Logistic, crude", exponentiate=True),
    tidy_model(adjusted_logit, "Logistic, adjusted", exponentiate=True),
])
results.to_csv(OUT_TABLES / "model_results.csv", index=False)
results[results["term"] == "adherence_10"].round(3)

## Key points

* State the estimand first; the method follows from the outcome and exposure types.
* Report effect sizes with 95% CIs. A p-value alone says nothing about size or clinical importance.
* Check the direction of every comparison (which group minus which).
* Adjust for confounders chosen from your framework, not for every variable you have.
* Odds ratios are not risk ratios when the outcome is common. Use predicted probabilities for risk.
* Running many tests and reporting only the significant ones is misleading. Plan your analyses in advance.